In [10]:
!ls -la /content/ESMFold/
!ls -la /content/AlphaFold2/

total 796
drwxr-xr-x 2 root root   4096 Oct  2 07:07 .
drwxr-xr-x 1 root root   4096 Oct  2 07:05 ..
-rw-r--r-- 1 root root 186292 Oct  2 07:06 ptm0.486_r3_default.pae.txt
-rw-r--r-- 1 root root 102507 Oct  2 07:06 ptm0.486_r3_default.pdb
-rw-r--r-- 1 root root 188608 Oct  2 07:06 ptm0.486_r3_default.png
-rw-r--r-- 1 root root  99935 Oct  2 07:07 ptm0.907_r3_default.pae.txt
-rw-r--r-- 1 root root  81042 Oct  2 07:07 ptm0.907_r3_default.pdb
-rw-r--r-- 1 root root 132982 Oct  2 07:07 ptm0.907_r3_default.png
total 5016
drwxr-xr-x 2 root root    4096 Oct  2 07:06 .
drwxr-xr-x 1 root root    4096 Oct  2 07:05 ..
-rw-r--r-- 1 root root    2696 Oct  2 07:06 cite.bibtex
-rw-r--r-- 1 root root    1267 Oct  2 07:06 config.json
-rw-r--r-- 1 root root    3026 Oct  2 07:06 log.txt
-rw-r--r-- 1 root root   99935 Oct  2 07:06 ptm0.907_r3_default.pae.txt
-rw-r--r-- 1 root root   81042 Oct  2 07:06 ptm0.907_r3_default.pdb
-rw-r--r-- 1 root root  132982 Oct  2 07:06 ptm0.907_r3_default.png
-rw-r--r-- 1 

In [11]:
!pip install biopython matplotlib numpy -q

In [12]:
import Bio

Считаем средний pLDDT по всем остаткам белка.

In [13]:
import os
import glob
from Bio.PDB import PDBParser

def get_mean_plddt(pdb_path, scale=1.0):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('model', pdb_path)
    plddts = []
    for model in structure:
        for chain in model:
            for residue in chain:
                if 'CA' in residue:
                    plddts.append(residue['CA'].get_bfactor() * scale) # CA (альфа-углерод) — это центральный атом каждой аминокислоты
        break
    return sum(plddts) / len(plddts) if plddts else 0.0

esm_files = {
    "test_ce6c6": "/content/ESMFold/ptm0.907_r3_default.pdb",
    "test_1a25a": "/content/ESMFold/ptm0.486_r3_default.pdb",
}

cf_files = {}
for p in glob.glob("/content/AlphaFold2/**/*.pdb", recursive=True):
    base = os.path.basename(p)
    if 'rank_001' in base:
        for name in ["test_ce6c6", "test_1a25a"]:
            if name in base:
                cf_files[name] = p

real_names = {
    "test_ce6c6": "1LYZ_HenEggWhiteLysozyme",
    "test_1a25a": "GLuc_GaussiaPrinceps",
}

print("Сопоставление файлов")
for name in ["test_ce6c6", "test_1a25a"]:
    real = real_names.get(name, name)
    print(f"{real}:")
    print(f"  ESMFold:   {esm_files.get(name, 'НЕ НАЙДЕН')}")
    print(f"  ColabFold: {cf_files.get(name, 'НЕ НАЙДЕН')}")
print()


print(f"{'Белок':<25} | {'ESMFold (0-100)':<18} | {'ColabFold (0-100)':<18} | Разница")
print("-" * 85)

for name in ["test_ce6c6", "test_1a25a"]:
    esm_pdb = esm_files.get(name)
    cf_pdb  = cf_files.get(name)
    real_name = real_names.get(name, name)

    if esm_pdb and cf_pdb and os.path.exists(esm_pdb) and os.path.exists(cf_pdb):
        esm_score = get_mean_plddt(esm_pdb, scale=1.0)
        cf_score  = get_mean_plddt(cf_pdb,  scale=1.0)
        diff = esm_score - cf_score
        print(f"{real_name:<25} | {esm_score:<18.2f} | {cf_score:<18.2f} | {diff:+.2f}")
    else:
        print(f"{real_name:<25} | Не найдено: ESM={esm_pdb}, CF={cf_pdb}")

=== Сопоставление файлов ===
1LYZ_HenEggWhiteLysozyme:
  ESMFold:   /content/ESMFold/ptm0.907_r3_default.pdb
  ColabFold: /content/AlphaFold2/test_ce6c6_unrelaxed_rank_001_alphafold2_ptm_model_5_seed_000.pdb
GLuc_GaussiaPrinceps:
  ESMFold:   /content/ESMFold/ptm0.486_r3_default.pdb
  ColabFold: /content/AlphaFold2/test_1a25a_unrelaxed_rank_001_alphafold2_ptm_model_2_seed_000.pdb

Белок                     | ESMFold (0-100)    | ColabFold (0-100)  | Разница
-------------------------------------------------------------------------------------
1LYZ_HenEggWhiteLysozyme  | 95.14              | 98.10              | -2.97
GLuc_GaussiaPrinceps      | 54.66              | 74.27              | -19.61


In [14]:
!pip install py3Dmol

Визуализируем PDB в py3Dmol с раскраской по pLDDT (B-factor). (красный (<50) → жёлтый → синий (>90))

In [15]:
import py3Dmol
import os

def view_pdb(pdb_path, label=""):
    with open(pdb_path, 'r') as f:
        pdb_data = f.read()

    view = py3Dmol.view(width=700, height=500)
    view.addModel(pdb_data, 'pdb')

    view.setStyle({
        'cartoon': {
            'colorscheme': {
                'prop': 'b',
                'gradient': 'roygb',
                'min': 50,
                'max': 100
            }
        }
    })

    view.zoomTo()
    print(f"{label}")
    print(f"  Файл: {pdb_path}")
    view.show()
    return view



# 1LYZ (Hen Egg White Lysozyme)
esm_1lyz = "/content/ESMFold/ptm0.907_r3_default.pdb"
cf_1lyz  = "/content/AlphaFold2/test_ce6c6_unrelaxed_rank_001_alphafold2_ptm_model_5_seed_000.pdb"

# GLuc (Gaussia Luciferase)
esm_gluc = "/content/ESMFold/ptm0.486_r3_default.pdb"
cf_gluc  = "/content/AlphaFold2/test_1a25a_unrelaxed_rank_001_alphafold2_ptm_model_2_seed_000.pdb"


print("=" * 60)
print("1LYZ (Hen Egg White Lysozyme)")
print("Средний pLDDT: ESMFold=95.14 | ColabFold=98.10")
print("=" * 60)

view_pdb(esm_1lyz, "1LYZ — ESMFold (pLDDT 95.14, pTM 0.907)")
view_pdb(cf_1lyz,  "1LYZ — ColabFold (pLDDT 98.10)")



print("\n" + "=" * 60)
print("GLuc (Gaussia Luciferase)")
print("Средний pLDDT: ESMFold=54.66 | ColabFold=74.27")
print("=" * 60)

view_pdb(esm_gluc, "GLuc — ESMFold (pLDDT 54.66, pTM 0.486)")
view_pdb(cf_gluc,  "GLuc — ColabFold (pLDDT 74.27)")

1LYZ (Hen Egg White Lysozyme)
Средний pLDDT: ESMFold=95.14 | ColabFold=98.10
1LYZ — ESMFold (pLDDT 95.14, pTM 0.907)
  Файл: /content/ESMFold/ptm0.907_r3_default.pdb


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

1LYZ — ColabFold (pLDDT 98.10)
  Файл: /content/AlphaFold2/test_ce6c6_unrelaxed_rank_001_alphafold2_ptm_model_5_seed_000.pdb


3Dmol.js failed to load for some reason. Please check your browser console for error messages.


GLuc (Gaussia Luciferase)
Средний pLDDT: ESMFold=54.66 | ColabFold=74.27
GLuc — ESMFold (pLDDT 54.66, pTM 0.486)
  Файл: /content/ESMFold/ptm0.486_r3_default.pdb


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

GLuc — ColabFold (pLDDT 74.27)
  Файл: /content/AlphaFold2/test_1a25a_unrelaxed_rank_001_alphafold2_ptm_model_2_seed_000.pdb


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

3Dmol.js failed to load for some reason. Please check your browser console for error messages.